In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [2]:
torch.manual_seed(42)

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [4]:
df = pd.read_csv('fashion-mnist_train.csv')
df.sample(5)

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
49151,0,0,0,0,0,0,0,0,41,124,...,86,13,0,0,0,0,0,0,0,0
25821,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
57173,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
35087,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
50590,6,0,0,0,0,0,0,0,1,1,...,33,0,0,0,192,78,9,0,0,0


In [5]:
X = df.iloc[:, 1:].values
y = df.iloc[:, 0].values


In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [7]:
X_train = X_train/255.0
X_test = X_test/255.0

In [8]:
from numpy import dtype
class CustomDataset(Dataset):

    def __init__(self, features, labels):

        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):

        return len(self.features)

    def __getitem__(self, index):

        return self.features[index], self.labels[index]

In [9]:
train_dataset = CustomDataset(X_train, y_train)

In [10]:
test_dataset = CustomDataset(X_test, y_test)

In [11]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, pin_memory=True)

In [12]:
class MyNN(nn.Module):

    def __init__(self, num_features):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(num_features, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(64, 10)
        )
    
    def forward(self, x):
        return self.model(x)

In [13]:
epochs = 100
learning_rate = 0.1

In [14]:
model = MyNN(X_train.shape[1])
model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.SGD(model.parameters(), lr=learning_rate, weight_decay=1e-4)

In [15]:
for epoch in range(epochs):

    total_epoch_loss = 0
    for batch_features, batch_labels in train_loader:

        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

        outputs = model(batch_features)

        loss = criterion(outputs, batch_labels)

        optimizer.zero_grad()
        loss.backward()

        optimizer.step()

        total_epoch_loss += loss.item()
    
    avg_loss = total_epoch_loss/len(train_loader)

    print(f"Epoch: {epoch+1}, Loss: {avg_loss}")




Epoch: 1, Loss: 0.6250404661397139
Epoch: 2, Loss: 0.4924874481658141
Epoch: 3, Loss: 0.45572176250318686
Epoch: 4, Loss: 0.4337390823364258
Epoch: 5, Loss: 0.4180022409657637
Epoch: 6, Loss: 0.40585135521988075
Epoch: 7, Loss: 0.3952388668656349
Epoch: 8, Loss: 0.38637320880095166
Epoch: 9, Loss: 0.3745098713984092
Epoch: 10, Loss: 0.3720112902422746
Epoch: 11, Loss: 0.36873586917916934
Epoch: 12, Loss: 0.3581008742401997
Epoch: 13, Loss: 0.3486653520266215
Epoch: 14, Loss: 0.3459769849081834
Epoch: 15, Loss: 0.3453836625367403
Epoch: 16, Loss: 0.3387857883647084
Epoch: 17, Loss: 0.33401473420361677
Epoch: 18, Loss: 0.33106795329848926
Epoch: 19, Loss: 0.3316668189490835
Epoch: 20, Loss: 0.3250290295183659
Epoch: 21, Loss: 0.32210945959885917
Epoch: 22, Loss: 0.3203149386793375
Epoch: 23, Loss: 0.3222919679482778
Epoch: 24, Loss: 0.3146567237029473
Epoch: 25, Loss: 0.31536918110400436
Epoch: 26, Loss: 0.3160052151679993
Epoch: 27, Loss: 0.3112092094918092
Epoch: 28, Loss: 0.3084125704

In [16]:
model.eval()

MyNN(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [17]:
total = 0
correct = 0

with torch.no_grad():

    for batch_features, batch_labels in test_loader:
        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
        outputs = model(batch_features)
        _, predicted = torch.max(outputs, 1)

        total += batch_labels.shape[0]
        correct += (predicted == batch_labels).sum().item()
    
print(correct/total)

0.8905833333333333


In [18]:
with torch.no_grad():

    for batch_features, batch_labels in train_loader:
        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
        outputs = model(batch_features)
        _, predicted = torch.max(outputs, 1)

        total += batch_labels.shape[0]
        correct += (predicted == batch_labels).sum().item()
    
print(correct/total)

0.9313666666666667
